# Gold Consumption Views

Build business-friendly Gold views over the governed dimensional model.

Outputs:
- `admission_journey`
- `icu_clinical_monitoring`
- `hospital_operational_activity`

These views provide the wide analytical surfaces used by Genie, dashboards and the Databricks App.

The underlying dimensions, facts and bridges remain the governed Gold data model.


In [0]:
from pyspark.sql import functions as F

GOLD_SCHEMA = "cdr_gold.analytics_mimic_iv"

print(f"Gold schema: {GOLD_SCHEMA}")

In [0]:
# Remove legacy physical marts if they still exist as tables

for name in [
    "admission_journey",
    "icu_clinical_monitoring",
    "hospital_operational_activity"
]:
    object_name = f"{GOLD_SCHEMA}.{name}"

    if spark.catalog.tableExists(object_name):
        object_type = spark.catalog.getTable(
            object_name
        ).tableType

        if object_type != "VIEW":
            spark.sql(
                f"DROP TABLE {object_name}"
            )

            print(
                f"Removed legacy table: {object_name}"
            )

## Admission Journey

Create one analysis-ready row per hospital admission by combining `fact_admission` with the conformed patient, careunit, service and diagnosis dimensions.

In [0]:
# Create governed admission journey view

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.admission_journey AS

SELECT
    -- Keys / grain
    p.subject_id,
    f.hadm_id,

    -- Patient
    p.gender,
    f.age_at_admission,
    f.age_group,

    -- Admission
    f.admittime,
    f.dischtime,
    CAST(f.admittime AS DATE) AS admission_date,
    YEAR(f.admittime) AS admission_year,
    MONTH(f.admittime) AS admission_month,
    f.admission_type,
    f.admission_location,
    f.discharge_location,

    -- Outcomes
    f.hospital_los_days,
    f.los_band,
    f.mortality_flag,

    -- Hospital journey
    first_cu.careunit_name AS first_careunit,
    last_cu.careunit_name AS last_careunit,
    f.transfer_count,
    f.distinct_careunit_count,

    -- Service
    COALESCE(initial_s.service_code, 'UNKNOWN') AS initial_service,
    COALESCE(final_s.service_code, 'UNKNOWN') AS final_service,
    f.service_change_count,
    COALESCE(initial_s.service_code, 'UNKNOWN') AS security_service,

    -- ICU
    f.icu_flag,
    f.icu_stay_count,
    f.total_icu_los_days,
    f.icu_los_complete,

    -- Clinical
    CASE
        WHEN is_account_group_member('healthcare_admins')
            OR is_account_group_member('healthcare_researchers')
        THEN COALESCE(dx.icd_code, 'UNKNOWN')
        ELSE 'RESTRICTED'
    END AS primary_diagnosis_code,

    CASE
        WHEN is_account_group_member('healthcare_admins')
            OR is_account_group_member('healthcare_researchers')
        THEN dx.icd_version
        ELSE NULL
    END AS primary_diagnosis_version,

    CASE
        WHEN is_account_group_member('healthcare_admins')
            OR is_account_group_member('healthcare_researchers')
        THEN COALESCE(dx.description, 'UNKNOWN')
        ELSE 'RESTRICTED'
    END AS primary_diagnosis,

    f.diagnosis_count,
    f.procedure_count,

    -- Audit
    f._gold_timestamp

FROM {GOLD_SCHEMA}.fact_admission f

LEFT JOIN {GOLD_SCHEMA}.dim_patient p
    ON f.patient_key = p.patient_key

LEFT JOIN {GOLD_SCHEMA}.dim_careunit first_cu
    ON f.first_careunit_key = first_cu.careunit_key

LEFT JOIN {GOLD_SCHEMA}.dim_careunit last_cu
    ON f.last_careunit_key = last_cu.careunit_key

LEFT JOIN {GOLD_SCHEMA}.dim_service initial_s
    ON f.initial_service_key = initial_s.service_key

LEFT JOIN {GOLD_SCHEMA}.dim_service final_s
    ON f.final_service_key = final_s.service_key

LEFT JOIN {GOLD_SCHEMA}.dim_diagnosis dx
    ON f.primary_diagnosis_key = dx.diagnosis_key

WHERE
    is_account_group_member('healthcare_admins')
 OR is_account_group_member('healthcare_hospital_operations')
 OR is_account_group_member('healthcare_researchers')
""")

print(f"Updated governed view: {GOLD_SCHEMA}.admission_journey")

In [0]:
admission_journey = spark.table(
    f"{GOLD_SCHEMA}.admission_journey"
)

display(
    admission_journey.agg(
        F.count("*").alias("rows"),
        F.countDistinct("hadm_id").alias("unique_admissions"),
        F.countDistinct("subject_id").alias("unique_patients"),
        F.sum("icu_flag").alias("icu_admissions"),
        F.sum("mortality_flag").alias("in_hospital_deaths")
    )
)

In [0]:
# Create governed ICU clinical monitoring view

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.icu_clinical_monitoring AS

SELECT
    -- Keys / grain
    f.subject_id,
    f.hadm_id,
    f.stay_id,
    f.clinical_date,
    f.icu_day_number,

    -- Patient / admission context
    p.gender,
    a.age_at_admission,
    a.age_group,
    a.admission_type,
    a.mortality_flag AS admission_mortality_flag,

    -- ICU context
    cu.careunit_name AS icu_careunit,
    cu.careunit_name AS security_careunit,
    f.icu_los_days,
    CAST(f.icu_stay_complete_flag AS BOOLEAN) AS icu_stay_complete,

    -- Vital monitoring
    f.vital_observation_count,
    f.distinct_vital_measurements,
    f.vital_warning_count,
    (f.vital_warning_count > 0) AS vital_warning_flag,

    f.heart_rate_avg,
    f.heart_rate_min,
    f.heart_rate_max,

    f.respiratory_rate_avg AS resp_rate_avg,
    f.respiratory_rate_min AS resp_rate_min,
    f.respiratory_rate_max AS resp_rate_max,

    f.spo2_avg,
    f.spo2_min,
    f.spo2_max,

    f.temperature_c_avg,
    f.temperature_c_min,
    f.temperature_c_max,

    -- Laboratory activity
    f.lab_result_count,
    f.distinct_lab_tests,
    f.abnormal_lab_count,
    f.abnormal_lab_flag,

    f.creatinine_avg,
    f.hemoglobin_avg,
    f.wbc_avg,
    f.sodium_avg,
    f.potassium_avg,
    f.lactate_avg,

    -- Fluid balance
    (f.input_event_count > 0) AS fluid_input_recorded,
    (f.output_event_count > 0) AS fluid_output_recorded,
    f.total_input_ml AS fluid_input_ml,
    f.total_output_ml AS fluid_output_ml,
    f.net_fluid_balance_ml,
    f.fluid_balance_complete,

    CASE
        WHEN NOT f.fluid_balance_complete THEN 'INCOMPLETE'
        WHEN f.net_fluid_balance_ml > 0 THEN 'POSITIVE'
        WHEN f.net_fluid_balance_ml < 0 THEN 'NEGATIVE'
        ELSE 'BALANCED'
    END AS fluid_balance_direction,

    -- Procedures
    f.procedure_count,
    f.distinct_procedure_types,

    -- Dataset-relative analytical flags
    f.high_vital_activity_flag AS high_vital_monitoring_flag,
    f.high_lab_activity_flag,
    f.high_procedure_activity_flag,
    f.large_fluid_shift_flag AS large_fluid_imbalance_flag,

    -- Audit
    f._gold_timestamp

FROM {GOLD_SCHEMA}.fact_icu_daily_monitoring f

LEFT JOIN {GOLD_SCHEMA}.dim_patient p
    ON f.patient_key = p.patient_key

LEFT JOIN {GOLD_SCHEMA}.fact_admission a
    ON f.hadm_id = a.hadm_id

LEFT JOIN {GOLD_SCHEMA}.dim_careunit cu
    ON f.careunit_key = cu.careunit_key

WHERE
    is_account_group_member('healthcare_admins')
 OR is_account_group_member('healthcare_researchers')
""")

print(f"Updated governed view: {GOLD_SCHEMA}.icu_clinical_monitoring")

In [0]:
# Create governed hospital operational activity view

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.hospital_operational_activity AS

SELECT
    -- Grain
    f.activity_date,
    f.careunit,

    -- Patient presence
    f.active_patient_count,
    f.distinct_admission_count,

    -- Movement
    f.patient_arrival_count,
    f.admission_arrival_count,
    f.transfer_arrival_count,
    f.patient_departure_count,
    f.net_patient_flow,
    f.total_patient_movement,
    f.daily_patient_turnover,

    -- ICU activity
    f.active_icu_stay_count,
    f.active_icu_patient_count,
    f.icu_activity_flag,

    -- Relative activity
    f.p95_active_patient_count,
    f.p95_total_patient_movement,
    f.high_patient_load_flag,
    f.high_movement_flag,
    f.high_activity_flag,

    -- Calendar
    YEAR(f.activity_date) AS activity_year,
    MONTH(f.activity_date) AS activity_month,
    WEEKOFYEAR(f.activity_date) AS activity_week,
    DATE_FORMAT(f.activity_date, 'EEEE') AS day_of_week,

    CASE
        WHEN DAYOFWEEK(f.activity_date) IN (1, 7) THEN 1
        ELSE 0
    END AS weekend_flag,

    -- Governance
    f.careunit AS security_careunit,

    -- Audit
    f._gold_timestamp

FROM {GOLD_SCHEMA}.fact_careunit_daily_activity f

WHERE
    is_account_group_member('healthcare_admins')
 OR is_account_group_member('healthcare_hospital_operations')
""")

print(f"Updated governed view: {GOLD_SCHEMA}.hospital_operational_activity")

In [0]:
admission_view = spark.table(
    f"{GOLD_SCHEMA}.admission_journey"
)

icu_view = spark.table(
    f"{GOLD_SCHEMA}.icu_clinical_monitoring"
)

operational_view = spark.table(
    f"{GOLD_SCHEMA}.hospital_operational_activity"
)

# Admission
display(
    admission_view.agg(
        F.count("*").alias("rows"),
        F.countDistinct("hadm_id").alias("unique_admissions"),
        F.countDistinct("subject_id").alias("unique_patients"),
        F.sum("icu_flag").alias("icu_admissions"),
        F.sum("mortality_flag").alias("in_hospital_deaths")
    )
)

# ICU
display(
    icu_view.agg(
        F.count("*").alias("rows"),
        F.countDistinct("stay_id").alias("unique_stays"),

        F.sum(
            (F.col("vital_observation_count") > 0).cast("int")
        ).alias("vital_days"),

        F.sum(
            (F.col("lab_result_count") > 0).cast("int")
        ).alias("lab_days"),

        F.sum(
            (F.col("procedure_count") > 0).cast("int")
        ).alias("procedure_days"),

        F.sum(
            F.col("fluid_balance_complete").cast("int")
        ).alias("complete_fluid_days")
    )
)

# Operational
display(
    operational_view.agg(
        F.count("*").alias("rows"),
        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit").alias("careunits"),

        F.sum("patient_arrival_count").alias("arrivals"),
        F.sum("patient_departure_count").alias("departures"),
        F.sum("high_activity_flag").alias("high_activity_days"),
        F.sum("icu_activity_flag").alias("icu_activity_days")
    )
)

In [0]:
# Show final Genie-facing view schemas

for view_name in [
    "admission_journey",
    "icu_clinical_monitoring",
    "hospital_operational_activity"
]:
    print(f"\n{view_name}")
    print("-" * len(view_name))

    for field in spark.table(
        f"{GOLD_SCHEMA}.{view_name}"
    ).schema.fields:
        print(field.name)